# joining everything together

we have 9 separate cleaned files. this notebook merges them into one master table
so every row = one order with its weather on that day

In [1]:
import pandas as pd
import numpy as np

# load all the cleaned files
orders        = pd.read_csv('cleaned/orders.csv', parse_dates=['order_purchase_timestamp', 'purchase_date'])
customers     = pd.read_csv('cleaned/customers.csv')
geo           = pd.read_csv('cleaned/geolocation.csv')
stations      = pd.read_csv('cleaned/weather_stations.csv')
weather       = pd.read_csv('cleaned/weather_daily.csv', parse_dates=['date'])
order_spend   = pd.read_csv('cleaned/order_spend.csv')
order_payments= pd.read_csv('cleaned/order_payments.csv')
order_cat     = pd.read_csv('cleaned/order_category.csv')

print('all loaded')

all loaded


## step 1 - attach customer location to each order

orders has customer_id -> customers has zip code -> geo has lat/lng for that zip

so we do two merges in a row to go from order to coordinates

In [2]:
# merge orders with customers on customer_id
df = orders.merge(customers[['customer_id', 'customer_zip_code_prefix', 'customer_state']], 
                  on='customer_id', how='left')

print('after adding customers:', df.shape)

# merge with geolocation on zip code to get lat/lng
df = df.merge(geo, left_on='customer_zip_code_prefix', right_on='zip_code_prefix', how='left')

# drop zip_code_prefix (duplicate of customer_zip_code_prefix)
df = df.drop(columns=['zip_code_prefix', 'state'])

print('after adding coordinates:', df.shape)
print('orders with no coordinates:', df['lat'].isnull().sum())
df.head(3)

after adding customers: (99441, 9)
after adding coordinates: (99441, 11)
orders with no coordinates: 278


,order_id,customer_id,order_status,order_purchase_timestamp,purchase_date,purchase_hour,is_returned,customer_zip_code_prefix,customer_state,lat,lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02,10,0,3149,SP,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-24,20,0,47813,BA,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08,8,0,75265,GO,-16.745150,-48.514783


## step 2 - find the nearest weather station for each zip code

we have coordinates for each customer and coordinates for each weather station.
for every zip code we want to find which station is closest.

we use a simple distance formula: sqrt((lat1-lat2)^2 + (lng1-lng2)^2) basically phythogorem
this is not 100% accurate over very long distances but its good enough

doint the calculation for each zip code rather than order

In [3]:
# get unique zip locations (no point calculating the same zip twice)
unique_zips = df[['customer_zip_code_prefix', 'lat', 'lng']].dropna().drop_duplicates('customer_zip_code_prefix')
print('unique zips to match:', len(unique_zips))

# station coordinates as numpy arrays for fast calculation
station_lats = stations['lat'].values
station_lngs = stations['lng'].values
station_ids  = stations['station_id'].values

nearest_station = []

for _, row in unique_zips.iterrows():
    # calculate distance from this zip to every station
    dist = np.sqrt((station_lats - row['lat'])**2 + (station_lngs - row['lng'])**2)
    # pick the station with the smallest distance
    closest = station_ids[np.argmin(dist)]
    nearest_station.append({'customer_zip_code_prefix': row['customer_zip_code_prefix'], 
                             'station_id': closest})

zip_to_station = pd.DataFrame(nearest_station)
print('zip to station mapping done, shape:', zip_to_station.shape)
zip_to_station.head(3)

unique zips to match: 14837


zip to station mapping done, shape: (14837, 2)


,customer_zip_code_prefix,station_id
0,3149.0,A701
1,47813.0,A402
2,75265.0,A037


In [4]:
# attach the nearest station id to each order
df = df.merge(zip_to_station, on='customer_zip_code_prefix', how='left')

print('orders with no station assigned:', df['station_id'].isnull().sum())
print(df.shape)

orders with no station assigned: 278
(99441, 12)


## step 3 - attach weather data

now each order has a station_id and a purchase_date.
we merge with weather_daily on those two columns to get the weather for that exact day

In [5]:
# make sure date types match before merging
df['purchase_date'] = pd.to_datetime(df['purchase_date'])
weather['date'] = pd.to_datetime(weather['date'])

df = df.merge(weather, left_on=['station_id', 'purchase_date'], 
              right_on=['station_id', 'date'], how='left')

# drop the duplicate date column that came from weather
df = df.drop(columns=['date'])

print('after adding weather:', df.shape)
print('orders with no weather data:', df['temp_mean_c'].isnull().sum())

after adding weather: (99441, 17)
orders with no weather data: 10083


## step 4 - attach spend and category

finally merge in how much was spent and what category was bought

In [6]:
df = df.merge(order_spend, on='order_id', how='left')
df = df.merge(order_payments[['order_id', 'total_payment_value', 'payment_type']], on='order_id', how='left')
df = df.merge(order_cat, on='order_id', how='left')

print('final shape:', df.shape)
print('columns:', df.columns.tolist())

final shape:

 (99441, 23)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'purchase_date', 'purchase_hour', 'is_returned', 'customer_zip_code_prefix', 'customer_state', 'lat', 'lng', 'station_id', 'temp_mean_c', 'temp_max_c', 'temp_min_c', 'precip_total_mm', 'humidity_mean', 'item_count', 'total_price', 'total_freight', 'total_payment_value', 'payment_type', 'main_category']


## step 5 - drop rows we cant use

if an order has no weather data we cant use it for our analysis so drop those
also drop orders with no spend data

In [7]:
before = len(df)

# drop rows with no temperature (means no weather data was found for that day/station)
df = df.dropna(subset=['temp_mean_c'])

# drop rows with no spend data
df = df.dropna(subset=['total_price'])

after = len(df)
print(f'dropped {before - after} rows, {after} remaining')
print('null counts:')
print(df.isnull().sum())

dropped 10766 rows, 88675 remaining
null counts:
order_id                       0
customer_id                    0
order_status                   0
order_purchase_timestamp       0
purchase_date                  0
purchase_hour                  0
is_returned                    0
customer_zip_code_prefix       0
customer_state                 0
lat                            0
lng                            0
station_id                     0
temp_mean_c                    0
temp_max_c                     0
temp_min_c                     0
precip_total_mm                0
humidity_mean                460
item_count                     0
total_price                    0
total_freight                  0
total_payment_value            1
payment_type                   1
main_category               1186
dtype: int64


In [8]:
df.to_csv('cleaned/master.csv', index=False)
print('saved master.csv')
df.head(5)

saved master.csv


,order_id,customer_id,order_status,order_purchase_timestamp,purchase_date,purchase_hour,is_returned,customer_zip_code_prefix,customer_state,lat,...,temp_max_c,temp_min_c,precip_total_mm,humidity_mean,item_count,total_price,total_freight,total_payment_value,payment_type,main_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02,10,0,3149,SP,-23.576983,...,22.7,16.8,16.8,63.958333,1.0,29.99,8.72,38.71,voucher,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-24,20,0,47813,BA,-12.177924,...,31.2,13.8,0.0,56.166667,1.0,118.70,22.76,141.46,boleto,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08,8,0,75265,GO,-16.745150,...,30.7,15.5,0.0,56.416667,1.0,159.90,19.22,179.12,credit_card,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18,19,0,59296,RN,-5.774190,...,29.0,25.4,1.0,73.833333,1.0,45.00,27.20,72.20,credit_card,pet_shop
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09,21,0,86320,PR,-23.553522,...,21.8,13.8,0.0,71.916667,1.0,147.90,27.36,175.26,credit_card,auto
